# 06 - Dashboard Usage Guide

Learn how to use the Flask-based real-time monitoring dashboard.

## Table of Contents
1. [Overview](#1.-Overview)
2. [Starting the Dashboard](#2.-Starting-the-Dashboard)
3. [Dashboard Features](#3.-Dashboard-Features)
4. [Loading Models and Data](#4.-Loading-Models-and-Data)
5. [Real-time Predictions](#5.-Real-time-Predictions)
6. [Visualization Features](#6.-Visualization-Features)
7. [Toolpath Visualization](#7.-Toolpath-Visualization)
8. [API Endpoints](#8.-API-Endpoints)
9. [Configuration Options](#9.-Configuration-Options)

---

## 1. Overview

The G-code Fingerprinting Dashboard v2.5 provides a real-time web interface for:

- **Live Predictions**: Stream sensor data and see predictions in real-time via WebSocket
- **Multi-Head Models**: Support for hierarchical token prediction (100% G-command accuracy)
- **Confusion Matrices**: Track prediction accuracy across 5 token heads
- **Sensor Visualization**: Heatmaps and time-series plots for all 232 sensors
- **Toolpath Visualization**: 3D GT vs Predicted path comparison with error coloring
- **Model Management**: Load different checkpoints with auto-detection
- **Data Export**: Download predictions as CSV or G-code (.nc files)

### Dashboard Architecture (v2.5)

```
┌─────────────────────────────────────────────────────────────────┐
│                   Flask Dashboard (port 4999)                    │
├─────────────────────────────────────────────────────────────────┤
│                                                                  │
│  ┌──────────────┐  ┌──────────────┐  ┌──────────────┐          │
│  │    Model     │  │   WebSocket  │  │  Prediction  │          │
│  │   Loader     │  │   Stream     │  │   Display    │          │
│  └──────────────┘  └──────────────┘  └──────────────┘          │
│                                                                  │
│  ┌──────────────┐  ┌──────────────┐  ┌──────────────┐          │
│  │  5-Head      │  │   Sensor     │  │   Toolpath   │          │
│  │  Confusion   │  │   Heatmap    │  │   3D Plot    │          │
│  └──────────────┘  └──────────────┘  └──────────────┘          │
│                                                                  │
│  ┌──────────────┐  ┌──────────────┐  ┌──────────────┐          │
│  │  Generation  │  │    Redis     │  │   Export     │          │
│  │  History     │  │   Cache      │  │   Tools      │          │
│  └──────────────┘  └──────────────┘  └──────────────┘          │
│                                                                  │
└─────────────────────────────────────────────────────────────────┘
```

### Key Features in v2.5

| Category | Features |
|----------|----------|
| **Core** | Token-level & full command predictions, 5-head confusion matrices |
| **Advanced** | Beam search, nucleus sampling, temperature control |
| **Toolpath** | 3D visualization, error gradient coloring, playback controls |
| **Export** | CSV predictions, G-code export (.nc files) |
| **Performance** | WebSocket streaming, Redis caching, batched updates |

In [ ]:
# ============================================================
# Environment Setup
# ============================================================

import sys
from pathlib import Path
import json
import time

import numpy as np
import requests

# Project root
project_root = Path.cwd().parent
sys.path.insert(0, str(project_root / 'src'))

# Dashboard Configuration - Updated to port 4999
DASHBOARD_URL = 'http://localhost:4999'

print("="*60)
print("G-CODE FINGERPRINTING DASHBOARD v2.5")
print("="*60)
print(f"Dashboard URL: {DASHBOARD_URL}")
print(f"Project Root: {project_root}")
print()
print("Key Features:")
print("  • Multi-Head Model Support (100% G-command accuracy)")
print("  • WebSocket real-time streaming")
print("  • Toolpath visualization with error coloring")
print("  • Redis caching for t-SNE & analytics")

## 2. Starting the Dashboard

Start the Flask dashboard in a separate terminal:

```bash
# Recommended: Run with PYTHONPATH set
PYTHONPATH=src .venv/bin/python flask_dashboard.py

# Alternative: Direct execution
cd /path/to/gcode_fingerprinting
.venv/bin/python flask_dashboard.py
```

The dashboard will:
1. Start on http://localhost:4999
2. Initialize SocketIO for real-time WebSocket updates
3. Connect to Redis for caching (if available)
4. Auto-detect available models (baseline vs multi-head)

### Startup Banner

When the dashboard starts, you'll see:
```
============================================================
🚀 Enhanced G-Code Fingerprinting Dashboard v2.5
============================================================

Core Features:
  ✅ Token-level & Full command predictions
  ✅ Live confusion matrix
  ✅ Dark mode & CSV export
  ✅ Sensor heatmap (all 232 sensors)
  ✅ 3D position tracking

Advanced Features:
  🔥 Beam search generation
  🔥 Toolpath visualization with error coloring
  🔥 Ground truth comparison & edit distance
  🔥 Generation history (last 50 commands)

Open: http://localhost:4999
============================================================
```

In [ ]:
# Check if dashboard is running
def check_dashboard_status():
    """Check if the dashboard is running."""
    try:
        response = requests.get(f'{DASHBOARD_URL}/', timeout=5)
        return response.status_code == 200
    except requests.exceptions.ConnectionError:
        return False
    except Exception:
        return False

is_running = check_dashboard_status()

if is_running:
    print("\n✓ Dashboard is running!")
    print(f"  Open in browser: {DASHBOARD_URL}")
else:
    print("\n✗ Dashboard is not running!")
    print("  Start with: PYTHONPATH=src .venv/bin/python flask_dashboard.py")

## 3. Dashboard Features

### Main Features

| Feature | Description |
|---------|-------------|
| **Model Selector** | Choose from available checkpoints (auto-detects multi-head vs baseline) |
| **CSV Upload** | Load sensor data from CSV files |
| **Live Predictions** | Real-time G-code prediction stream via WebSocket |
| **5-Head Confusion Matrix** | Per-head accuracy: operation, type, command, param_type, param_value |
| **Sensor Heatmap** | Visualize all 232 sensor activation patterns |
| **3D Position Plot** | Track predicted XYZ movements |
| **Toolpath Visualization** | Compare GT vs Predicted paths with error coloring |
| **Generation History** | Last 50 commands with ground truth comparison |
| **Dark Mode** | Toggle dark/light theme |
| **CSV/G-code Export** | Download prediction results or .nc files |

### Model Types

| Model Type | Description | Accuracy |
|------------|-------------|----------|
| **Baseline** | Single-head LSTM decoder | ~85% full command |
| **Multi-Head** | 4-head hierarchical decoder | 100% G-command, 92%+ full |
| **Sensor Multi-Head** | Multi-head with sensor decoder | Same + sensor reconstruction |

### Prediction Heads (Multi-Head Models)

```python
PREDICTION_HEADS = {
    'operation': ['G', 'M', 'T', 'S', 'F', ...],      # G-code operation (100% acc)
    'type': ['0', '1', '2', '3', '17', '54', ...],    # Type number
    'command': ['G0', 'G1', 'G2', 'M30', ...],        # Full command token
    'param_type': ['X', 'Y', 'Z', 'F', 'S', ...],     # Parameter type
    'param_value': ['-1.234', '0.0', '100', ...],     # Parameter value
}
```

### Operation Types

The dashboard tracks 9 CNC operation classes:

```python
OPERATION_TYPES = [
    "adaptive",        # 0
    "adaptive150025",  # 1
    "face",            # 2
    "face150025",      # 3
    "pocket",          # 4
    "pocket150025",    # 5
    "damageadaptive",  # 6
    "damageface",      # 7
    "damagepocket",    # 8
]
```

## 4. Loading Models and Data

### Available Models

In [ ]:
# Get available models from dashboard API
def get_available_models():
    """Fetch list of available models from dashboard."""
    try:
        response = requests.get(f'{DASHBOARD_URL}/api/models', timeout=5)
        if response.status_code == 200:
            return response.json()
        return None
    except Exception as e:
        print(f"Error: {e}")
        return None

models = get_available_models()
if models:
    print("\nAvailable Models:")
    print("-" * 50)
    for model in models.get('models', [])[:5]:
        print(f"  • {model}")
else:
    print("\nCould not fetch models (dashboard may not be running)")

In [ ]:
# Get available CSV files
def get_available_csv_files():
    """Fetch list of available CSV data files."""
    try:
        response = requests.get(f'{DASHBOARD_URL}/api/csv_files', timeout=5)
        if response.status_code == 200:
            return response.json()
        return None
    except Exception as e:
        print(f"Error: {e}")
        return None

csv_files = get_available_csv_files()
if csv_files:
    print("\nAvailable CSV Files:")
    print("-" * 50)
    for csv_file in csv_files.get('files', [])[:5]:
        print(f"  • {csv_file}")
else:
    print("\nCould not fetch CSV files")

In [ ]:
# Load a model programmatically
def load_model(checkpoint_name):
    """Load a specific model checkpoint."""
    payload = {'checkpoint': checkpoint_name}
    try:
        response = requests.post(
            f'{DASHBOARD_URL}/api/load_model',
            json=payload,
            timeout=30
        )
        return response.json()
    except Exception as e:
        return {'error': str(e)}

# Example usage (uncomment to test)
# result = load_model('checkpoint_best.pt')
# print(f"Load result: {result}")

print("\nTo load a model programmatically:")
print("  result = load_model('checkpoint_name.pt')")

## 5. Real-time Predictions

The dashboard uses WebSocket (SocketIO) for real-time streaming.

In [ ]:
# SocketIO client example
try:
    from socketio import Client
    SOCKETIO_AVAILABLE = True
except ImportError:
    SOCKETIO_AVAILABLE = False
    print("python-socketio not installed. Install with: pip install python-socketio")

if SOCKETIO_AVAILABLE:
    print("\nSocketIO Client Example:")
    print("-" * 50)
    print("""
from socketio import Client

sio = Client()

@sio.on('prediction_update')
def on_prediction(data):
    '''Handle real-time prediction updates.'''
    print(f"Prediction: {data['full_command']}")
    print(f"Ground Truth: {data['ground_truth']}")
    print(f"Edit Distance: {data['edit_distance']}")
    
    # Multi-head confidences
    for head, conf in data.get('confidences', {}).items():
        print(f"  {head}: {conf:.2%}")

@sio.on('toolpath_update')
def on_toolpath(data):
    '''Handle toolpath visualization updates.'''
    point = data['point']
    print(f"Point {data['total_points']}: error={point['error_distance']:.4f}mm")
    print(f"  GT:   ({point['gt']['x']:.3f}, {point['gt']['y']:.3f}, {point['gt']['z']:.3f})")
    print(f"  Pred: ({point['pred']['x']:.3f}, {point['pred']['y']:.3f}, {point['pred']['z']:.3f})")

@sio.on('confusion_matrix_update')
def on_confusion(data):
    '''Handle confusion matrix updates.'''
    for head, matrix in data.items():
        print(f"Updated {head} confusion matrix")

# Connect and start streaming
sio.connect('http://localhost:4999')
sio.emit('start_inference')
sio.wait()  # Wait for events
    """)

# WebSocket Events Reference
print("\nWebSocket Events Reference:")
print("="*50)
events = {
    'connect': 'Connection established',
    'disconnect': 'Connection closed',
    'start_inference': 'Begin streaming predictions (client → server)',
    'stop_inference': 'Pause streaming (client → server)',
    'prediction_update': 'Real-time prediction (server → client)',
    'toolpath_update': 'Toolpath point update (server → client)',
    'confusion_matrix_update': 'Matrix update (server → client)',
}
for event, desc in events.items():
    print(f"  {event:25s} {desc}")

In [ ]:
# Simulate prediction stream visualization
import matplotlib.pyplot as plt

# Generate simulated prediction data
np.random.seed(42)
n_steps = 50

# Simulated confidence scores over time
type_conf = 0.85 + 0.1 * np.random.randn(n_steps).cumsum() * 0.01
type_conf = np.clip(type_conf, 0.6, 1.0)

command_conf = 0.75 + 0.1 * np.random.randn(n_steps).cumsum() * 0.01
command_conf = np.clip(command_conf, 0.5, 0.95)

param_type_conf = 0.80 + 0.1 * np.random.randn(n_steps).cumsum() * 0.01
param_type_conf = np.clip(param_type_conf, 0.6, 0.95)

param_value_conf = 0.65 + 0.1 * np.random.randn(n_steps).cumsum() * 0.01
param_value_conf = np.clip(param_value_conf, 0.4, 0.85)

# Plot
fig, ax = plt.subplots(figsize=(12, 5))

ax.plot(type_conf, label='Type', linewidth=2)
ax.plot(command_conf, label='Command', linewidth=2)
ax.plot(param_type_conf, label='Param Type', linewidth=2)
ax.plot(param_value_conf, label='Param Value', linewidth=2)

ax.set_xlabel('Time Step')
ax.set_ylabel('Confidence')
ax.set_title('Simulated Real-time Prediction Confidence')
ax.legend(loc='lower right')
ax.set_ylim(0.3, 1.05)
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print("\nThis visualization simulates the real-time confidence display in the dashboard.")

## 6. Visualization Features

### Sensor Heatmaps

In [ ]:
# Fetch sensor heatmap data
def get_sensor_heatmap():
    """Fetch sensor vs time heatmap data."""
    try:
        response = requests.get(f'{DASHBOARD_URL}/api/heatmap/sensors_vs_time', timeout=10)
        if response.status_code == 200:
            return response.json()
        return None
    except Exception as e:
        return None

# Example heatmap visualization
print("\nSensor Heatmap Visualization:")
print("-" * 50)

# Generate example heatmap
np.random.seed(42)
n_sensors = 20
n_timesteps = 64

# Simulate sensor data with patterns
sensor_data = np.zeros((n_sensors, n_timesteps))
for i in range(n_sensors):
    freq = 0.1 + i * 0.02
    phase = i * 0.5
    sensor_data[i] = np.sin(np.linspace(0, 4*np.pi, n_timesteps) * freq + phase)
    sensor_data[i] += 0.2 * np.random.randn(n_timesteps)

fig, ax = plt.subplots(figsize=(14, 6))

im = ax.imshow(sensor_data, aspect='auto', cmap='viridis')
ax.set_xlabel('Time Step')
ax.set_ylabel('Sensor Channel')
ax.set_title('Sensor Activation Heatmap (Example)')
plt.colorbar(im, label='Activation')

plt.tight_layout()
plt.show()

print("\nThe dashboard displays this heatmap in real-time as data streams in.")

In [ ]:
# Confusion matrix visualization
print("\nConfusion Matrix Example:")
print("-" * 50)

# Generate example confusion matrix
from sklearn.metrics import confusion_matrix
import seaborn as sns

# Simulated predictions for 9 operation classes
np.random.seed(42)
n_samples = 500
n_classes = 9

# Create somewhat realistic predictions (high accuracy with some confusion)
true_labels = np.random.randint(0, n_classes, n_samples)
pred_labels = true_labels.copy()

# Add some errors (15% error rate)
error_mask = np.random.random(n_samples) < 0.15
pred_labels[error_mask] = np.random.randint(0, n_classes, error_mask.sum())

cm = confusion_matrix(true_labels, pred_labels)

# Normalize
cm_normalized = cm.astype('float') / cm.sum(axis=1)[:, np.newaxis]

# Operation type names
op_names = ['adapt', 'adpt150', 'face', 'face150', 'pocket', 'pkt150', 'dmg_adp', 'dmg_fce', 'dmg_pkt']

fig, ax = plt.subplots(figsize=(10, 8))

sns.heatmap(
    cm_normalized,
    annot=True,
    fmt='.2f',
    cmap='Blues',
    xticklabels=op_names,
    yticklabels=op_names,
    ax=ax
)

ax.set_xlabel('Predicted')
ax.set_ylabel('True')
ax.set_title('Operation Type Confusion Matrix (Normalized)')

plt.tight_layout()
plt.show()

accuracy = np.diag(cm).sum() / cm.sum()
print(f"\nOverall Accuracy: {accuracy:.2%}")

## 7. Toolpath Visualization

The dashboard includes a dedicated tab for comparing Ground Truth vs Predicted G-code toolpaths in 3D.

### View Modes

| Mode | Description |
|------|-------------|
| **Error-Colored** | Single path colored by position error (green→yellow→red) |
| **Overlaid** | GT (blue) and Predicted (orange) paths overlaid |
| **Side-by-Side** | Two synchronized 3D plots with linked cameras |

### Features

- **Continuous Error Gradient**: HSL color mapping from green (0 error) to red (max error)
- **Motion Type Styling**: G0 (dashed/rapid), G1 (solid), G2/G3 (thick arcs)
- **Playback Controls**: Play, pause, step forward/backward, jump to error
- **Accuracy Timeline**: Clickable bar chart showing error at each command
- **Command Comparison**: Side-by-side GT vs Predicted with diff highlighting
- **Performance Throttling**: Max 10 updates/sec with level-of-detail decimation

### Error Color Scale

```
Error Distance    Color
─────────────────────────
0.0mm             Green (HSL 120°)
0.05mm            Yellow-Green (HSL 60°)
0.1mm+            Red (HSL 0°)
```

In [ ]:
# Toolpath API Example
print("Toolpath Visualization API:")
print("="*50)

# Fetch toolpath data
def get_toolpath_data():
    """Fetch accumulated toolpath data."""
    try:
        response = requests.get(f'{DASHBOARD_URL}/api/toolpath/accumulated', timeout=10)
        if response.status_code == 200:
            return response.json()
        return None
    except Exception as e:
        return {'error': str(e)}

# Fetch toolpath metrics
def get_toolpath_metrics():
    """Fetch toolpath accuracy metrics."""
    try:
        response = requests.get(f'{DASHBOARD_URL}/api/toolpath/state', timeout=5)
        if response.status_code == 200:
            data = response.json()
            return data.get('metrics', {})
        return None
    except Exception:
        return None

# Example: Parse G-code to coordinates
print("\nG-code to Coordinates Example:")
print("-" * 50)
print("""
# Use the G-code parser directly
from miracle.utilities.gcode_parser import GCodeToCoordinateParser

parser = GCodeToCoordinateParser(arc_segments=10)
gcode_sequence = ['G1 X10.0 Y20.0 Z-0.5', 'G1 X15.0 Y25.0', 'G0 Z5.0']
points = parser.parse_sequence(gcode_sequence)

for pt in points:
    print(f"  ({pt.x:.2f}, {pt.y:.2f}, {pt.z:.2f}) - {pt.motion_type}")
""")

# Toolpath API endpoints
print("\nToolpath API Endpoints:")
print("-" * 50)
toolpath_endpoints = {
    'GET /api/toolpath/parse': 'Parse G-code string to coordinates',
    'GET /api/toolpath/accumulated': 'Get all accumulated GT vs Pred points',
    'GET /api/toolpath/range': 'Get points in index range',
    'GET /api/toolpath/point/{idx}': 'Get specific comparison point',
    'GET /api/toolpath/errors': 'Get all error distances',
    'GET /api/toolpath/state': 'Get current toolpath state and metrics',
}
for endpoint, desc in toolpath_endpoints.items():
    print(f"  {endpoint:35s} {desc}")

# Dashboard API reference (Updated for v2.5)
api_endpoints = {
    # Core endpoints
    'GET /': 'Main dashboard page',
    'GET /api/status': 'Get inference status and model info',
    'GET /api/models': 'List available model checkpoints',
    'GET /api/csv_files': 'List available CSV data files',
    'POST /api/load_model': 'Load a specific model checkpoint',
    'POST /api/load_csv': 'Load a CSV file for streaming',
    
    # Prediction & Generation
    'GET /api/generation_history': 'Get last 50 predictions with GT comparison',
    'GET /api/command_types': 'Get command type distribution',
    'POST /api/settings': 'Update generation settings (temp, beam, etc.)',
    'GET /api/export_gcode': 'Export predictions as .nc file',
    
    # Confusion Matrices
    'GET /api/confusion_matrix': 'Get all confusion matrices',
    'GET /api/confusion_matrix/{head}/normalized': 'Get normalized matrix for head',
    
    # Visualizations
    'GET /api/heatmap/sensors_vs_time': 'Get sensor heatmap data',
    'GET /api/tsne': 'Get t-SNE embedding visualization',
    
    # Toolpath (NEW)
    'GET /api/toolpath/accumulated': 'Get all GT vs Pred comparison points',
    'GET /api/toolpath/state': 'Get toolpath metrics and machine state',
    'GET /api/toolpath/errors': 'Get error distance array',
}

print("\nDashboard API Endpoints (v2.5):")
print("="*60)

for endpoint, description in api_endpoints.items():
    parts = endpoint.split(' ', 1)
    method = parts[0]
    path = parts[1] if len(parts) > 1 else ''
    print(f"  {method:6s} {path:40s} {description}")

In [ ]:
# Dashboard API reference
api_endpoints = {
    'GET /': 'Main dashboard page',
    'GET /api/models': 'List available model checkpoints',
    'GET /api/csv_files': 'List available CSV data files',
    'POST /api/load_model': 'Load a specific model checkpoint',
    'POST /api/load_csv': 'Load a CSV file for streaming',
    'GET /api/heatmap/sensors_vs_time': 'Get sensor heatmap data',
    'GET /api/heatmap/sensors_vs_vibration': 'Get vibration correlation heatmap',
}

print("\nDashboard API Endpoints:")
print("="*60)

for endpoint, description in api_endpoints.items():
    method, path = endpoint.split(' ', 1)
    print(f"  {method:6s} {path:40s} {description}")

In [ ]:
# Test all endpoints
def test_dashboard_endpoints():
    """Test all dashboard API endpoints."""
    results = {}
    
    endpoints = [
        ('GET', '/'),
        ('GET', '/api/models'),
        ('GET', '/api/csv_files'),
    ]
    
    for method, path in endpoints:
        try:
            if method == 'GET':
                response = requests.get(f'{DASHBOARD_URL}{path}', timeout=5)
            status = response.status_code
            results[path] = '✓' if status == 200 else f'✗ ({status})'
        except Exception as e:
            results[path] = f'✗ ({str(e)[:20]})'
    
    return results

print("\nEndpoint Health Check:")
print("-" * 40)

results = test_dashboard_endpoints()
for endpoint, status in results.items():
    print(f"  {endpoint:30s} {status}")

# Default generation settings (v2.5)
default_settings = {
    'enable_autoregressive': False,  # Disabled by default for performance
    'max_tokens': 15,
    'temperature': 1.0,
    'top_p': 1.0,
    'beam_size': 1,
    'use_beam_search': False,
    'inference_delay': 0.1,          # Delay between samples (seconds)
}

print("\nGeneration Settings (v2.5):")
print("="*50)

for key, value in default_settings.items():
    print(f"  {key:25s} {value}")

print("\n" + "="*50)
print("Configuration Tips:")
print("-" * 50)
print("""
• enable_autoregressive: Enable for better quality (slower)
• max_tokens: Maximum tokens to generate per sample
• temperature: Higher = more random, Lower = more deterministic  
• top_p: Nucleus sampling threshold (0.9 = top 90% probability)
• beam_size: Number of beams (1-5, higher = better but slower)
• use_beam_search: Enable beam search (requires beam_size > 1)
• inference_delay: Seconds between samples (0.05 - 1.0)
""")

# Example: Update settings via API
print("\nUpdate Settings via API:")
print("-" * 50)
print("""
import requests

settings = {
    'temperature': 0.8,
    'beam_size': 3,
    'use_beam_search': True
}

response = requests.post(
    'http://localhost:4999/api/settings',
    json=settings
)
print(response.json())
""")

In [ ]:
# Dashboard startup checklist
print("\nDashboard Startup Checklist:")
print("="*50)

checklist = [
    ('Virtual environment activated', '.venv/bin/activate'),
    ('PYTHONPATH set', 'export PYTHONPATH=src'),
    ('Model checkpoint available', 'outputs/*/best_model.pt'),
    ('Vocabulary file exists', 'data/gcode_vocab_v2.json'),
    ('Port 4999 available', 'lsof -i :4999'),
    ('Redis running (optional)', 'redis-cli ping'),
]

for item, command in checklist:
    print(f"  [ ] {item}")
    print(f"      Check: {command}")

print("\n" + "-"*50)
print("Start command:")
print("  PYTHONPATH=src .venv/bin/python flask_dashboard.py")
print()
print("Verify it's running:")
print("  curl http://localhost:4999/api/status")

In [ ]:
## Summary

In this notebook, you learned:

- **Dashboard architecture**: Flask + SocketIO for real-time WebSocket updates
- **Starting the dashboard**: Launch command and configuration on port 4999
- **Key features**: Model loading, multi-head predictions, toolpath visualization
- **API endpoints**: REST API for programmatic access (20+ endpoints)
- **Visualizations**: Heatmaps, confusion matrices, 3D toolpaths, confidence plots
- **Toolpath visualization**: GT vs Predicted comparison with error coloring

### Dashboard URL

Once running, access the dashboard at:

```
http://localhost:4999
```

### Key Commands

```bash
# Start dashboard
PYTHONPATH=src .venv/bin/python flask_dashboard.py

# Check if running
curl http://localhost:4999/api/status

# View available models
curl http://localhost:4999/api/models

# Stop dashboard
Ctrl+C in terminal
```

### Quick Reference

| Feature | Access |
|---------|--------|
| Main Dashboard | http://localhost:4999 |
| API Status | http://localhost:4999/api/status |
| Model List | http://localhost:4999/api/models |
| Toolpath Data | http://localhost:4999/api/toolpath/accumulated |

---

**Navigation:**
← [Previous: 05_api_usage](05_api_usage.ipynb) |
[Next: 07_hyperparameter_sweeps](07_hyperparameter_sweeps.ipynb) →

**Related:** [08_model_evaluation](08_model_evaluation.ipynb) | [10_visualization_experiments](10_visualization_experiments.ipynb)

## Summary

In this notebook, you learned:

- **Dashboard architecture**: Flask + SocketIO for real-time updates
- **Starting the dashboard**: Launch command and configuration
- **Key features**: Model loading, predictions, visualizations
- **API endpoints**: REST API for programmatic access
- **Visualizations**: Heatmaps, confusion matrices, confidence plots

### Dashboard URL

Once running, access the dashboard at:

```
http://localhost:5000
```

### Key Commands

```bash
# Start dashboard
PYTHONPATH=src .venv/bin/python flask_dashboard.py

# Check if running
curl http://localhost:5000/api/models

# Stop dashboard
Ctrl+C in terminal
```

---

**Navigation:**
← [Previous: 05_api_usage](05_api_usage.ipynb) |
[Next: 07_hyperparameter_sweeps](07_hyperparameter_sweeps.ipynb) →

**Related:** [08_model_evaluation](08_model_evaluation.ipynb) | [10_visualization_experiments](10_visualization_experiments.ipynb)